In [1]:
import numpy as np
import torch
import tqdm

from pathlib import Path

from src.manifolds.sn_mfld import HypersphereManifold
from src.episodes.traj import generate_hs_trajectory, Trajectory
from src.episodes.episode import Episode

rand = np.random.default_rng(42)  # for reproducibility

In [2]:
HYPERSPHERE_MIN_DIM = 3
HYPERSPHERE_MAX_DIM = 3
HYPERSPHERE_RADIUS = 1.0

NUM_TRAJECTORIES_PER_DIM = 1
NUM_INITIAL_POINTS_PER_TRAJECTORY = 11

TRAJECTORY_INITIAL_COORD_DIST = (0.0, np.pi / 4)
TRAJECTORY_WAYPOINT_COORD_DIST = (0.0, np.pi / 4)
TRAJECTORY_WAYPOINT_TRAVEL_TIME_DIST = (30.0, 0.0)

TRAJECTORY_NUM_WAYPOINTS = 2
TRAJECTORY_SAMPLE_TIME = 0.05

EPISODE_INITIAL_POS_STD = 0.4
EPISODE_INITIAL_VEL_DIST = (0, 0.2)

EPISODE_DATA_DIR = Path("../../../data/episodes/hypersphere")

In [3]:
def _project_point_onto_hypersphere(x: np.ndarray, radius: float) -> np.ndarray:
    return radius * x / np.linalg.norm(x)


def _project_vel_onto_hypersphere(x: np.ndarray, dot_x: np.ndarray, radius: float) -> np.ndarray:
    return radius * (dot_x / np.linalg.norm(x) - np.dot(x, dot_x) / np.linalg.norm(x) ** 3 * x)

In [4]:
episodes = []
for dim in tqdm.tqdm(range(HYPERSPHERE_MIN_DIM, HYPERSPHERE_MAX_DIM + 1), desc="Hypersphere dim", total=HYPERSPHERE_MAX_DIM):
    ambient_dim = dim + 1

    # matches the distributino to the dimension of the ambient space
    traj_initial_pos_dist = (TRAJECTORY_INITIAL_COORD_DIST[0] * np.ones((ambient_dim,)),  # mean
                             TRAJECTORY_INITIAL_COORD_DIST[1] ** 2 * np.eye(ambient_dim))  # std
    traj_waypoint_pos_dist = (TRAJECTORY_WAYPOINT_COORD_DIST[0] * np.ones((ambient_dim,)),  # mean
                              TRAJECTORY_WAYPOINT_COORD_DIST[1] ** 2 * np.eye(ambient_dim))  # std
    episode_initial_pos_covar = EPISODE_INITIAL_POS_STD ** 2 * np.eye(ambient_dim)  # covariance
    episode_initial_vel_dist = (EPISODE_INITIAL_VEL_DIST[0] * np.ones((ambient_dim,)),  # mean
                                EPISODE_INITIAL_VEL_DIST[1] ** 2 * np.eye(ambient_dim))  # std

    hypersphere_manifold = HypersphereManifold(dim, radius=HYPERSPHERE_RADIUS)

    for traj_idx in range(NUM_TRAJECTORIES_PER_DIM):
        # this is a randomized starting position in the ambient space (which will eventually be projected onto the hypersphere)
        traj_initial_pos = rand.multivariate_normal(*traj_initial_pos_dist)
        traj: Trajectory = generate_hs_trajectory(
            pos_start_ambient=traj_initial_pos,
            waypoint_dist=traj_waypoint_pos_dist,
            waypoint_travel_time_dist=TRAJECTORY_WAYPOINT_TRAVEL_TIME_DIST,
            num_waypoints=TRAJECTORY_NUM_WAYPOINTS,
            dt=TRAJECTORY_SAMPLE_TIME,
            radius=HYPERSPHERE_RADIUS,
            coord_sys=hypersphere_manifold,
            rand=rand,
        )

        # now generates multiple episodes from the trajectory (which is basically just the trajectory but with different starting positions which the control law will regulate
        for start_idx in range(NUM_INITIAL_POINTS_PER_TRAJECTORY):
            traj_initial_pos = traj.extrinsic[0][0, :]
            traj_initial_vel = traj.extrinsic[1][0, :]

            # extrinsic initial episode trajectory (distributed around the sphere)

            print(f"COVAR: {episode_initial_pos_covar.shape}")

            episode_initial_pos = rand.multivariate_normal(traj_initial_pos, episode_initial_pos_covar)
            episode_initial_vel = rand.multivariate_normal(*episode_initial_vel_dist)

            # project the initial episode trajectory back onto the hypersphere
            episode_initial_pos = _project_point_onto_hypersphere(episode_initial_pos, radius=HYPERSPHERE_RADIUS)
            episode_initial_vel = _project_vel_onto_hypersphere(episode_initial_vel, dot_x=episode_initial_vel,
                                                                radius=HYPERSPHERE_RADIUS)

            episode = Episode(traj, episode_initial_pos, episode_initial_vel, params={
                "hs_radius": HYPERSPHERE_RADIUS,
                "hs_dim": dim,
                "sample_time": TRAJECTORY_SAMPLE_TIME,
                "traj_idx": traj_idx,
                "episode_idx": start_idx,
            })
            episodes.append(
                (f"hs_dim_{dim}_radius_{HYPERSPHERE_RADIUS}_traj_{traj_idx}_episode_{start_idx}".replace(".", "_"),
                 episode))

Hypersphere dim:   0%|          | 0/3 [00:00<?, ?it/s]

x: (1200, 4), dot_x: (1200, 4)
norm: (1200, 1)
hypersphere_x_term_1: (1200, 4)
hypersphere_x_term_2: (1200, 4)


Hypersphere dim:  33%|███▎      | 1/3 [00:40<01:21, 40.78s/it]

COVAR: (4, 4)
COVAR: (4, 4)
COVAR: (4, 4)
COVAR: (4, 4)
COVAR: (4, 4)
COVAR: (4, 4)
COVAR: (4, 4)
COVAR: (4, 4)
COVAR: (4, 4)
COVAR: (4, 4)
COVAR: (4, 4)


In [5]:
for name, episode in tqdm.tqdm(episodes, desc="Episodes", total=len(episodes)):
    episode.save(EPISODE_DATA_DIR / f"{name}.npz")

Episodes: 100%|██████████| 11/11 [00:00<00:00, 1131.95it/s]
